In [1]:
import tensorflow as tf
from tensorflow.keras.layers import GRU,Dense,Dropout # type: ignore
from tensorflow.keras.models import Sequential # type: ignore
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler

In [2]:
data = pd.read_csv('jena_climate_2009_2016.csv',index_col=['Date Time'])
data.index = pd.to_datetime(data.index,format='%d.%m.%Y %H:%M:%S')
data = data[(data.index.minute == 0) & (data.index.hour % 3 == 0)]

tempc = data['T (degC)']
var = data.drop(columns=['T (degC)','Tpot (K)'])
data.head(10)

,p (mbar),T (degC),Tpot (K),Tdew (degC),rh (%),VPmax (mbar),VPact (mbar),VPdef (mbar),sh (g/kg),H2OC (mmol/mol),rho (g/m**3),wv (m/s),max. wv (m/s),wd (deg)
Date Time,,,,,,,,,,,,,,
2009-01-01 03:00:00,996.84,-8.81,264.59,-9.66,93.5,3.13,2.93,0.20,1.83,2.94,1312.18,0.18,0.63,167.20
2009-01-01 06:00:00,997.71,-9.67,263.66,-10.62,92.7,2.93,2.71,0.21,1.69,2.72,1317.71,0.05,0.50,146.00
2009-01-01 09:00:00,999.69,-7.66,265.52,-8.84,91.2,3.43,3.13,0.30,1.95,3.13,1310.14,0.34,0.63,202.20
2009-01-01 12:00:00,1000.30,-6.87,266.27,-8.28,89.6,3.64,3.27,0.38,2.03,3.26,1306.98,1.84,2.63,184.40
2009-01-01 15:00:00,999.88,-5.69,267.48,-7.00,90.4,3.99,3.61,0.38,2.25,3.61,1300.51,1.17,1.88,134.90
2009-01-01 18:00:00,1000.16,-5.25,267.90,-6.75,89.1,4.13,3.68,0.45,2.29,3.68,1298.68,0.55,1.00,183.70
2009-01-01 21:00:00,1000.19,-4.80,268.35,-6.14,90.2,4.27,3.85,0.42,2.40,3.85,1296.45,0.44,0.75,206.30
2009-01-02 00:00:00,999.59,-4.54,268.65,-5.46,93.2,4.36,4.06,0.30,2.53,4.06,1294.33,0.41,0.88,155.00
2009-01-02 03:00:00,998.69,-4.45,268.81,-5.15,94.8,4.39,4.16,0.23,2.59,4.16,1292.69,0.65,1.00,203.30


In [3]:
# Normalization
scalar = StandardScaler()
var = scalar.fit_transform(var)

seq_len = 56
step = 2
sequences = []
targets = []

for i in range(0,len(data)-seq_len-7,step):
    seq  = var[i:i+seq_len]
    target = tempc[i+seq_len+7]
    sequences.append(seq)
    targets.append(target)

sequences, targets  = np.array(sequences), np.array(targets)

x_train,x_validation,x_test = sequences[:8000], sequences[8000:9500], sequences[9500:]
y_train,y_validation,y_test = targets[:8000], targets[8000:9500], targets  [9500:]

C:\Users\Ben\AppData\Local\Temp\ipykernel_17856\744585239.py:12: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  target = tempc[i+seq_len+7]


In [ ]:
model = Sequential()
model.add(GRU(units=128,activation='tanh',recurrent_dropout=0.3,input_shape=(56,12),return_sequences=True)) # on ajoute return_sequences pour avoir le vecteur de la sortie qui est y pour chaque unité  
model.add(GRU(units=128,activation='tanh',recurrent_dropout=0.3,dropout=0.3)) # dans cett couche on fait recurrent_dropout sur les poinds de toutes les unités et un dropout sur la rentrée de cette couche 
model.add(Dropout(0.5)) # la sortie (y) qui est un vecteur de la couche précedante (GRU) sera dopout de 0.3 
model.add(Dense(128,activation='relu'))
model.add(Dropout(0.5))
model.add(Dense(1,activation='linear'))

model.compile(optimizer= tf.optimizers.Adam(learning_rate=0.001), loss= tf.losses.mean_absolute_error)

model.fit(x_train, y_train, epochs=10, batch_size=256, validation_data=(x_validation,y_validation), verbose=2)


Epoch 1/10

32/32 - 29s - loss: 6.9053 - val_loss: 3.9920 - 29s/epoch - 904ms/step
Epoch 2/10
32/32 - 20s - loss: 4.2647 - val_loss: 3.1657 - 20s/epoch - 626ms/step
Epoch 3/10
32/32 - 20s - loss: 3.3664 - val_loss: 2.6052 - 20s/epoch - 630ms/step
Epoch 4/10
32/32 - 20s - loss: 3.0849 - val_loss: 2.4678 - 20s/epoch - 620ms/step
Epoch 5/10
32/32 - 22s - loss: 3.0448 - val_loss: 2.4580 - 22s/epoch - 703ms/step
Epoch 6/10
32/32 - 29s - loss: 2.9630 - val_loss: 2.4601 - 29s/epoch - 893ms/step
Epoch 7/10
32/32 - 28s - loss: 2.9708 - val_loss: 2.4528 - 28s/epoch - 889ms/step
Epoch 8/10
32/32 - 30s - loss: 2.9274 - val_loss: 2.5473 - 30s/epoch - 942ms/step
Epoch 9/10
32/32 - 35s - loss: 2.9103 - val_loss: 2.4376 - 35s/epoch - 1s/step
Epoch 10/10
32/32 - 35s - loss: 2.9015 - val_loss: 2.4786 - 35s/epoch - 1s/step
